In [1]:
 %run /Users/manojrammopati/Downloads/bupa_insurance_project/00_Pre_Pilot/00_Pre_Pilot/_00__Pre_Connectors.ipynb

25/11/28 13:25:08 WARN Utils: Your hostname, Manojrams-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 192.168.0.219 instead (on interface en0)
25/11/28 13:25:08 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /Users/manojrammopati/.ivy2/cache
The jars for the packages stored in: /Users/manojrammopati/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-azure added as a dependency
com.azure#azure-storage-blob added as a dependency
com.azure#azure-identity added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-e65909ec-59db-48f2-9e8f-30d1a498a360;1.0
	confs: [default]


:: loading settings :: url = jar:file:/opt/anaconda3/envs/spark_local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found io.delta#delta-spark_2.12;3.1.0 in central
	found io.delta#delta-storage;3.1.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.hadoop#hadoop-azure;3.3.4 in central
	found org.apache.httpcomponents#httpclient;4.5.13 in central
	found org.apache.httpcomponents#httpcore;4.4.13 in central
	found commons-logging#commons-logging;1.1.3 in central
	found commons-codec#commons-codec;1.15 in central
	found com.microsoft.azure#azure-storage;7.0.1 in central
	found com.fasterxml.jackson.core#jackson-core;2.12.7 in central
	found org.slf4j#slf4j-api;1.7.36 in central
	found com.microsoft.azure#azure-keyvault-core;1.0.0 in central
	found com.google.guava#guava;27.0-jre in central
	found com.google.guava#failureaccess;1.0 in central
	found com.google.guava#listenablefuture;9999.0-empty-to-avoid-conflict-with-guava in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found org.checkerframework#checker-qual;2.5.2 in central
	found com.google.errorpron

Preconnectors ran successfully and connected to 'clientdatastorage' storage accounts


In [43]:
bronze_layer = "abfss://rawdata@clientdatastorage.dfs.core.windows.net/"
policy_df = (
    spark.read.format("delta")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(f"{bronze_layer}policies")
)

In [38]:
print("Policies :", policy_df.count())
policy_df.show(5, truncate=False)

Policies : 381109


+----------+------------+------------+------------------+------------------+-----------------+---------------+--------------------+---------------------+------------------+-------+
|Policy_ID |Customer_ID |Product_Line|Sum_Insured_GBP   |Annual_Premium_GBP|Policy_Start_Date|Policy_End_Date|Renewal_Offered_Flag|Renewal_Accepted_Flag|Discount_Offered_%|Channel|
+----------+------------+------------+------------------+------------------+-----------------+---------------+--------------------+---------------------+------------------+-------+
|P_00000001|CUST_0000001|Dental      |2193748.339982436 |40454.0           |2025-10-12       |2026-05-05     |1                   |0                    |3.418994275911136 |Partner|
|P_00000002|CUST_0000002|Health      |1908192.4592739474|33536.0           |2025-06-04       |2026-08-23     |1                   |1                    |16.10228791601092 |Partner|
|P_00000003|CUST_0000003|Health      |2356477.1034685415|38294.0           |2022-01-18       |2

In [44]:
policy_df.describe().show()

+-------+----------+------------+------------+--------------------+------------------+--------------------+---------------------+------------------+-------+
|summary| Policy_ID| Customer_ID|Product_Line|     Sum_Insured_GBP|Annual_Premium_GBP|Renewal_Offered_Flag|Renewal_Accepted_Flag|Discount_Offered_%|Channel|
+-------+----------+------------+------------+--------------------+------------------+--------------------+---------------------+------------------+-------+
|  count|    381109|      381109|      381109|              381109|            381109|              381109|               381109|            381109| 381109|
|   mean|      NULL|        NULL|        NULL|  1757359.9134775458|30564.389581458323|  0.7489694549328407|   0.6987397306282455| 7.492448600214125|   NULL|
| stddev|      NULL|        NULL|        NULL|  1000337.1978572499|17213.155056979947|  0.4336066233883118|  0.45880613750613414| 6.619117468444391|   NULL|
|    min|P_00000001|CUST_0000001|    Accident|   131501.53

In [30]:
policy_df.dtypes

[('Policy_ID', 'string'),
 ('Customer_ID', 'string'),
 ('Product_Line', 'string'),
 ('Sum_Insured_GBP', 'double'),
 ('Annual_Premium_GBP', 'double'),
 ('Policy_Start_Date', 'date'),
 ('Policy_End_Date', 'date'),
 ('Renewal_Offered_Flag', 'int'),
 ('Renewal_Accepted_Flag', 'int'),
 ('Discount_Offered_%', 'double'),
 ('Channel', 'string')]

In [39]:
claim_df = (
    spark.read.format("delta")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(f"{bronze_layer}claims")
)

In [47]:
print("claims :", claim_df.count())
claim_df.show(5, truncate=False)

claims : 558211


+--------+-----------+----------+-------------+------------+----------+------------------+-----------+----------+------------+
|Claim_ID|Provider_ID|Member_Key|Date_Reported|Date_Settled|Payout_GBP|Claim_Amount_GBP  |Fraud_Label|Claim_Type|Claim_Status|
+--------+-----------+----------+-------------+------------+----------+------------------+-----------+----------+------------+
|CLM46614|PRV55912   |BENE11001 |NULL         |NULL        |26000.0   |30397.346951642197|0          |Hospital  |Settled     |
|CLM66048|PRV55907   |BENE11001 |NULL         |NULL        |5000.0    |6379.96457105169  |0          |Hospital  |Settled     |
|CLM68358|PRV56046   |BENE11001 |NULL         |NULL        |5000.0    |7307.291499809586 |0          |Hospital  |Settled     |
|CLM38412|PRV52405   |BENE11011 |NULL         |NULL        |5000.0    |6989.424384581965 |0          |Hospital  |Settled     |
|CLM63689|PRV56614   |BENE11014 |NULL         |NULL        |10000.0   |13123.302569965643|0          |Hospital 

In [40]:
claim_df.describe().show()

+-------+---------+-----------+----------+-----------------+------------------+--------------------+----------+------------+
|summary| Claim_ID|Provider_ID|Member_Key|       Payout_GBP|  Claim_Amount_GBP|         Fraud_Label|Claim_Type|Claim_Status|
+-------+---------+-----------+----------+-----------------+------------------+--------------------+----------+------------+
|  count|   558211|     519069|    558211|           558211|            558211|              558211|    519015|      558211|
|   mean|     NULL|       NULL|      NULL|997.0121334047519|1295.3144682763134|0.015325745999272677|      NULL|        NULL|
| stddev|     NULL|       NULL|      NULL|3821.534891392819| 5005.326329393253| 0.12284500210928345|      NULL|        NULL|
|    min|CLM110011|   PRV51001|BENE100000|              0.0|               0.0|                   0|    Dental|     Pending|
|    max| CLM82318|   PRV57763| BENE99999|         125000.0|194526.70737420116|                   1|    Travel|   Withdrawn|


In [48]:
claim_df.dtypes

[('Claim_ID', 'string'),
 ('Provider_ID', 'string'),
 ('Member_Key', 'string'),
 ('Date_Reported', 'date'),
 ('Date_Settled', 'date'),
 ('Payout_GBP', 'double'),
 ('Claim_Amount_GBP', 'double'),
 ('Fraud_Label', 'int'),
 ('Claim_Type', 'string'),
 ('Claim_Status', 'string')]

In [49]:
member_df = (
    spark.read.format("delta")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(f"{bronze_layer}members")
)

In [50]:
print("members :", member_df.count())
member_df.show(5, truncate=False)

members : 381109


+------------+----------+---+------+------------------+------+---------------+-----------------+------+
|Member_ID   |Policy_ID |Age|Gender|BMI               |Smoker|Chronic_Disease|Employment_Status|Region|
+------------+----------+---+------+------------------+------+---------------+-----------------+------+
|MEM_00000001|P_00000001|44 |Male  |29.136545138305703|Y     |Hypertension   |Employed         |280   |
|MEM_00000002|P_00000002|76 |Male  |28.692615840826242|N     |None           |Employed         |30    |
|MEM_00000003|P_00000003|47 |Male  |23.118112520150454|Y     |Asthma         |Retired          |280   |
|MEM_00000004|P_00000004|21 |Male  |22.422099668655306|N     |Asthma         |Employed         |110   |
|MEM_00000005|P_00000005|29 |Female|31.74705333410654 |N     |Hypertension   |Employed         |410   |
+------------+----------+---+------+------------------+------+---------------+-----------------+------+
only showing top 5 rows



In [51]:
member_df.describe().show()

+-------+------------+----------+------------------+------+------------------+------+---------------+-----------------+------------------+
|summary|   Member_ID| Policy_ID|               Age|Gender|               BMI|Smoker|Chronic_Disease|Employment_Status|            Region|
+-------+------------+----------+------------------+------+------------------+------+---------------+-----------------+------------------+
|  count|      381109|    381109|            381109|381109|            381109|381109|         381109|           381109|            381109|
|   mean|        NULL|      NULL|38.822583565331705|  NULL|27.502522433095358|  NULL|           NULL|             NULL|263.88807401557034|
| stddev|        NULL|      NULL| 15.51161101809548|  NULL|3.4637336858733705|  NULL|           NULL|             NULL|132.29888025788586|
|    min|MEM_00000001|P_00000001|                20|Female| 21.50004160963284|     N|         Asthma|         Employed|                00|
|    max|MEM_00381109|P_003

In [52]:
member_df.dtypes

[('Member_ID', 'string'),
 ('Policy_ID', 'string'),
 ('Age', 'int'),
 ('Gender', 'string'),
 ('BMI', 'double'),
 ('Smoker', 'string'),
 ('Chronic_Disease', 'string'),
 ('Employment_Status', 'string'),
 ('Region', 'string')]

In [53]:
provider_df = (
    spark.read.format("delta")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(f"{bronze_layer}providers")
)

In [54]:
print("providers :", provider_df.count())
provider_df.show(5, truncate=False)

providers : 5410


+-----------+----------+
|Provider_ID|Fraud_Flag|
+-----------+----------+
|PRV51262   |0         |
|PRV51606   |0         |
|PRV51657   |0         |
|PRV51665   |0         |
|PRV51749   |0         |
+-----------+----------+
only showing top 5 rows



In [55]:
provider_df.describe().show()

+-------+-----------+-------------------+
|summary|Provider_ID|         Fraud_Flag|
+-------+-----------+-------------------+
|  count|       5410|               5410|
|   mean|       NULL|0.09353049907578559|
| stddev|       NULL| 0.2912013378502412|
|    min|   PRV51001|                  0|
|    max|   PRV57763|                  1|
+-------+-----------+-------------------+



In [56]:
provider_df.dtypes

[('Provider_ID', 'string'), ('Fraud_Flag', 'int')]

25/11/25 05:08:50 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 1073318 ms exceeds timeout 120000 ms
25/11/25 05:08:50 WARN SparkContext: Killing executors is not supported by current scheduler.
25/11/25 05:08:54 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManager(BlockManagerMaster.scala:80)
	at org.apache.spark.storage.BlockManager.reregister(BlockManager.scala:642)
	at org.apache.spark.executor.Executor.reportHeartBeat(Executor.scala:1223)
	at 

# no need of this below code cells ///

In [ ]:
# ============================
# BRONZE INGESTION NOTEBOOK
# ============================
spark.sql("CREATE DATABASE IF NOT EXISTS bupa_bronze")

base_path = "abfss://bronze@bupaprocesseddatastorage.dfs.core.windows.net/"

# --- 2. Write to Delta ---
policy_df.write.format("delta").mode("overwrite").save(f"{base_path}/delta_tables/policies")
member_df.write.format("delta").mode("overwrite").save(f"{base_path}/delta_tables/members")
claim_df.write.format("delta").mode("overwrite").save(f"{base_path}/delta_tables/claims")
provider_df.write.format("delta").mode("overwrite").save(f"{base_path}/delta_tables/providers")

print("✅ Bronze Delta tables created.")

In [ ]:
MAGIC %sql
CREATE DATABASE IF NOT EXISTS bupa_bronze;

In [ ]:
display(claim_df)
display(policy_df)
display(member_df)
display(provider_df)

In [ ]:
## For renaming the folder in ADLS GEN2 Using dbutils.fs.mv

dbutils.fs.mv(
    "abfss://bronze@bipclientstorage.dfs.core.windows.net/claims/bupa_claims_data.csv",
    "abfss://bronze@bipclientstorage.dfs.core.windows.net/claims/bupa_claims_data",
    recurse=True
)

In [ ]:
# Make sure the database exists
spark.sql("CREATE DATABASE IF NOT EXISTS bupa_bronze")

# Save DataFrames as tables in the database
claim_df.write.format("delta").mode("overwrite").saveAsTable("bupa_bronze.claims")
member_df.write.format("delta").mode("overwrite").saveAsTable("bupa_bronze.members")
policy_df.write.format("delta").mode("overwrite").saveAsTable("bupa_bronze.policies")
provider_df.write.format("delta").mode("overwrite").saveAsTable("bupa_bronze.providers")

print("✅ Tables are registered and created in Hive_Metastore successfully.")

In [ ]:
claims_table_df = spark.read.table("bupa_bronze.claims")
display(claims_table_df)

In [ ]:
MAGIC %sql
select * from bupa_bronze.policies
limit(10)

In [ ]:
MAGIC %sql
select * from bupa_bronze.members
limit(10)

In [ ]:
MAGIC %sql
select * from bupa_bronze.claims
limit(10)

In [ ]:
MAGIC %sql
select * from bupa_bronze.providers
limit(10)